## D . Dask : Parallelisation of the image processing algorithm Median Filter

-> **Expected Results:**
    
    - Dask parallel median_filter.py
    
    - Execute on lena_noizy.jpg and generate lena_filter.jpg

import imageio.v2 as imageio
import os
import numpy as np
import dask.array as da
from dask import delayed, compute
from dask.distributed import Client

In [2]:
%pip install imageio

Note: you may need to restart the kernel to use updated packages.


In [4]:
def readImg(path):
    img = imageio.imread(path)
    im = np.array(img, dtype='uint8')
    return im

def writeImg(path, buf):
    imageio.imwrite(path, buf)

def part_median_filter(part_buf):
    """Apply median filter to a part of the image."""
    height, width, channels = part_buf.shape
    filtered_part = np.zeros((height, width, channels), dtype='uint8')
    
    # Apply median filter to each channel separately
    for c in range(channels):
        for i in range(1, height - 1):
            for j in range(1, width - 1):
                # Extract the 3x3 neighborhood for the current pixel
                neighborhood = part_buf[i-1:i+2, j-1:j+2, c]
                # Compute the median and assign it to the filtered pixel
                filtered_part[i, j, c] = np.median(neighborhood)
    
    return filtered_part

In [5]:
def main():
    data_dir = './data'
    file = os.path.join(data_dir, 'lena_noisy.jpg')
    img_buf = readImg(file)
    print('SHAPE', img_buf.shape)
    print('IMG\n', img_buf)
    nx = img_buf.shape[0]
    ny = img_buf.shape[1]
    
    ###########################################################################
    #
    # SPLIT IMAGES INTO NB_PARTITIONS PARTS WITH OVERLAP
    nb_partitions = 8
    print("NB PARTITIONS : ", nb_partitions)
    block_size = nx // nb_partitions
    overlap = 2  # Overlap by 2 pixels
    
    # Split the image into overlapping parts
    parts = []
    for ip in range(nb_partitions):
        start = max(0, ip * block_size - overlap)
        end = min(nx, (ip + 1) * block_size + overlap)
        part = img_buf[start:end, :, :]
        parts.append(part)
    
    ###########################################################################
    #
    # CREATE DASK CLIENT
    client = Client()  # Start a Dask distributed client
    print("Dask Dashboard:", client.dashboard_link)
    
    ###########################################################################
    #
    # PARALLEL MEDIAN FILTER COMPUTATION USING DASK
    delayed_results = []
    for part in parts:
        # Use Dask's delayed to parallelize the median filter computation
        delayed_part = delayed(part_median_filter)(part)
        delayed_results.append(delayed_part)
    
    # Compute the results in parallel
    filtered_parts = compute(*delayed_results)
    
    ###########################################################################
    #
    # RECONSTRUCT THE IMAGE FROM THE FILTERED PARTS
    new_img_buf = np.zeros((nx, ny, 3), dtype='uint8')
    for ip, filtered_part in enumerate(filtered_parts):
        start = ip * block_size
        end = min((ip + 1) * block_size, nx)
        
        # Remove overlap before placing the filtered part into the final image
        if ip == 0:
            new_img_buf[start:end, :, :] = filtered_part[:-overlap, :, :]
        elif ip == nb_partitions - 1:
            new_img_buf[start:end, :, :] = filtered_part[overlap:, :, :]
        else:
            new_img_buf[start:end, :, :] = filtered_part[overlap:-overlap, :, :]
    
    ###########################################################################
    #
    # SAVE THE FILTERED IMAGE
    print('CREATE NEW PICTURE FILE')
    filter_file = os.path.join(data_dir, 'lena_filter_dask.jpg')
    writeImg(filter_file, new_img_buf)
    
    ###########################################################################
    #
    # CLOSE DASK CLIENT
    client.close()

if __name__ == '__main__':
    main()

SHAPE (128, 128, 3)
IMG
 [[[233 159 122]
  [228 159 118]
  [223 165 115]
  ...
  [181 116  98]
  [231 165 153]
  [223 156 147]]

 [[220 155 127]
  [226 166 132]
  [232 179 139]
  ...
  [199 134 116]
  [205 139 125]
  [162  98  86]]

 [[207 161 148]
  [206 160 144]
  [254 209 188]
  ...
  [156  93  75]
  [134  71  56]
  [114  52  37]]

 ...

 [[ 88  52  56]
  [155 119 123]
  [109  68  76]
  ...
  [131  78 108]
  [104  55  74]
  [107  61  72]]

 [[ 93  62  70]
  [219 186 193]
  [ 95  56  59]
  ...
  [100  56  71]
  [118  82  96]
  [122  90 105]]

 [[ 89  62  71]
  [ 81  50  56]
  [ 86  47  50]
  ...
  [109  69  77]
  [125  94 109]
  [214 190 206]]]
NB PARTITIONS :  8
Dask Dashboard: http://127.0.0.1:8787/status
CREATE NEW PICTURE FILE
